# TP 1 : les moindres carrés et le ridge

Ajuster un modèle linéaire à la main, puis regarder ce que deux mesures presque identiques font
à ses coefficients.

## 1.0 Deux estimateurs sur les mêmes patients

442 patients diabétiques, dix mesures prises au départ, et une cible : la progression de la
maladie un an plus tard, de 25 à 346.

Le même modèle linéaire s'ajuste de deux façons. Le **maximum de vraisemblance** n'écoute que les
données et se ramène aux moindres carrés. Le **maximum a posteriori** y ajoute la croyance que les
coefficients sont petits, et se ramène au ridge. Ce TP les construit toutes les deux à la main, sur
ces patients, et compare ce qu'elles rendent.

Deux des dix mesures se ressemblent beaucoup : `s1` le cholestérol total et `s2` le LDL, qui en est
une composante. On verra que cette ressemblance décide de presque tout ce qui suit.

Les colonnes sont `age`, `sex`, `bmi` (indice de masse corporelle), `bp` (tension), puis six
analyses sanguines `s1` à `s6`. **`s1` est le cholestérol total, `s2` le LDL** ; et le LDL est
une composante du cholestérol total.

Chaque colonne a été centrée puis divisée par sa norme : elles valent toutes entre $-0{,}2$ et
$0{,}2$, avec un écart-type de $1/\sqrt{442} \approx 0{,}048$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_linnerud
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

data = load_diabetes(as_frame=True)
feature_names = list(data.data.columns)
X, y = data.data.values, data.target.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"{len(X_train)} patients d'entraînement, {len(X_test)} de test, "
      f"{X.shape[1]} mesures : {', '.join(feature_names)}")
data.data.describe().loc[["min", "max", "std"]].round(3)

## 1.1 L'équation normale

Le modèle prédit un score linéaire. Pour ne pas traîner le biais $w_0$ dans toutes les formules,
on ajoute aux données une colonne fictive qui vaut toujours $1$ :

$$h_w(x) = w^T x = w_0 \cdot 1 + w_1 x_1 + \dots + w_n x_n,
  \qquad x = (1, x_1, \dots, x_n) \in \mathbb{R}^{n+1}.$$

On empile les $m$ entrées augmentées comme lignes de la **matrice de conception**
$X \in \mathbb{R}^{m \times (n+1)}$, et on cherche le $w$ qui minimise la somme des carrés :

$$J(w) = \sum_{i=1}^{m} \big(w^T x^{(i)} - y^{(i)}\big)^2 = \lVert Xw - y \rVert^2 .$$

Annuler $\nabla_w J = 2X^T(Xw - y)$ donne l'**équation normale** :

$$\boxed{\ X^T X\, w = X^T y, \qquad w = (X^T X)^{-1} X^T y\ }$$

On la résout comme un système linéaire, avec `np.linalg.solve(A, b)`, plutôt qu'en inversant
$X^T X$ puis en multipliant.

**Q1.** `X_train` a 353 lignes et 10 colonnes, et on ajoute devant une colonne qui vaut
toujours $1$. Combien de nombres le modèle apprend-il ? Ce compte changerait-il avec un million de
patients ?

**Votre réponse :**



In [ ]:
def design_matrix(X):
    """Add the dummy feature x_0 = 1 as the first column."""
    # TODO: return X with a leading column of ones, shape (m, n + 1)
    ...


def normal_equation(X, y):
    """Least-squares weights, bias first."""
    A = design_matrix(X)
    # TODO: solve (A^T A) w = A^T y with np.linalg.solve, and return w
    ...


w = normal_equation(X_train, y_train)
reference = LinearRegression().fit(X_train, y_train)
assert design_matrix(X_train).shape == (353, 11)
assert np.allclose(design_matrix(X_train)[:, 0], 1.0)
assert np.isclose(w[0], reference.intercept_)
assert np.allclose(w[1:], reference.coef_)


def predict(X, w):
    # TODO: return the predictions of w on X
    ...


def cost(X, y, w):
    """J(w), the sum of squared errors."""
    # TODO: return the sum of the squared differences between the predictions and y
    ...


def r2(y, y_hat):
    # TODO: 1 - sum of squared residuals / sum of squared deviations from the mean of y
    ...


assert np.allclose(predict(X_test, w), reference.predict(X_test))
assert np.isclose(r2(y_test, predict(X_test, w)), reference.score(X_test, y_test))
# the normal equation is exactly the w that minimises J: nudging any weight can only raise it
nudged = w.copy()
nudged[3] += 1.0
assert cost(X_train, y_train, w) < cost(X_train, y_train, nudged)

print(f"R² d'entraînement {r2(y_train, predict(X_train, w)):.3f}, "
      f"R² de test {r2(y_test, predict(X_test, w)):.3f}")
print(f"coût d'entraînement J(w) = {cost(X_train, y_train, w):,.0f}".replace(",", " "))
print()
print(pd.DataFrame({"coefficient": w[1:]}, index=feature_names).round(1).to_string())

correlations = pd.DataFrame(X_train, columns=feature_names).corr()
print(f"\ncorrélation entre s1 (cholestérol total) et s2 (LDL) : {correlations.loc['s1', 's2']:.3f}")

## 1.2 Rétrécir sans annuler

On ajoute au coût une pénalité sur la taille des poids :

$$J(w) = \lVert Xw - y \rVert^2 + \lambda \lVert w_{1:n} \rVert^2 ,$$

ce qui déplace l'équation normale d'un multiple de l'identité :

$$\boxed{\ w = (X^T X + \lambda I)^{-1} X^T y\ }$$

C'est la **régression ridge**. Le biais $w_0$ n'est pas pénalisé (rien ne justifie de tirer vers
zéro le niveau moyen de la cible), donc le coin $(0,0)$ de $I$ vaut $0$.

**Q2.** L'a priori se résume à « avant de voir les données, je crois les coefficients petits », et
il ajoute au coût la somme de leurs carrés. Les moindres carrés rendent des coefficients allant
jusqu'à 732 : que va leur faire cette pénalité ?

**Votre réponse :**



In [ ]:
def ridge_equation(X, y, lam):
    """Ridge weights, bias first. The bias is not penalised."""
    A = design_matrix(X)
    penalty = lam * np.eye(A.shape[1])
    penalty[0, 0] = 0.0
    # TODO: solve (A^T A + penalty) w = A^T y
    ...


w_ridge = ridge_equation(X_train, y_train, 1.0)
reference_ridge = Ridge(alpha=1.0).fit(X_train, y_train)
assert np.isclose(w_ridge[0], reference_ridge.intercept_)
assert np.allclose(w_ridge[1:], reference_ridge.coef_)
assert np.allclose(ridge_equation(X_train, y_train, 0.0), w)

comparison = pd.DataFrame({"moindres carrés": w[1:], "ridge λ = 1": w_ridge[1:]},
                          index=feature_names)
print(comparison.round(0).to_string())

print(f"\nR² de test : moindres carrés {r2(y_test, predict(X_test, w)):.3f}, "
      f"ridge {r2(y_test, predict(X_test, w_ridge)):.3f}")

**Q3.** Sous la pénalité, le coefficient de `s1` passe de $-663$ à 3, tandis que `bmi` ne perd que
la moitié du sien. Qu'est-ce qui distingue ces deux mesures, pour que la pénalité les traite si
différemment ?

**Votre réponse :**



## 1.3 Choisir $\lambda$

$\lambda$ ne s'apprend pas sur les données d'entraînement : à $\lambda = 0$ le coût
d'entraînement est toujours le plus bas. C'est un **hyperparamètre**, et on le choisit sur une
partie mise de côté, la **validation**, découpée dans l'entraînement ; le test reste fermé.

**Q4.** On va tracer l'erreur de validation pour $\lambda$ allant de $10^{-4}$ à $10^{4}$.
Va-t-elle monter sans arrêt, ou descendre puis remonter, et pourquoi ?

**Votre réponse :**



In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=0)
lambdas = np.logspace(-4, 4, 40)


def rmse(X, y, w):
    # TODO: root mean squared error of the weights w on (X, y)
    ...


paths, rmse_fit, rmse_val = [], [], []
for lam in lambdas:
    # TODO: fit on (X_fit, y_fit) with this lam, then append its 10 coefficients to paths,
    # its training RMSE to rmse_fit and its validation RMSE to rmse_val
    ...

paths = np.array(paths)
assert paths.shape == (40, 10)
assert np.allclose(paths[0], ridge_equation(X_fit, y_fit, lambdas[0])[1:])
assert np.isclose(rmse(X_test, y_test, w), np.sqrt(np.mean((predict(X_test, w) - y_test) ** 2)))

best = lambdas[int(np.argmin(rmse_val))]
print(f"λ retenu sur la validation : {best:.4g}")
print(f"RMSE de validation : {min(rmse_val):.2f} à ce λ, "
      f"{rmse_val[0]:.2f} à λ = {lambdas[0]:.4g}, {rmse_val[-1]:.2f} à λ = {lambdas[-1]:.0f}")
print()
for lam in (0.0, best, 1.0, 10.0, 100.0):
    print(f"λ = {lam:<8.4g} R² de test {r2(y_test, predict(X_test, ridge_equation(X_train, y_train, lam))):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for j, name in enumerate(feature_names):
    axes[0].plot(lambdas, paths[:, j], label=name)
axes[0].set_xscale("log")
axes[0].axvline(best, color="crimson", ls="--", lw=1)
axes[0].set_xlabel("λ")
axes[0].set_ylabel("coefficient")
axes[0].legend(fontsize=7, ncol=2)
axes[1].plot(lambdas, rmse_fit, "o-", ms=3, label="RMSE d'entraînement")
axes[1].plot(lambdas, rmse_val, "o-", ms=3, label="RMSE de validation")
axes[1].set_xscale("log")
axes[1].axvline(best, color="crimson", ls="--", lw=1)
axes[1].set_xlabel("λ")
axes[1].legend()
plt.tight_layout()
plt.show()

**Q5.** Sur la figure de droite, la RMSE d'entraînement ne fait que monter avec $\lambda$.
Pourquoi ne peut-elle pas servir à choisir $\lambda$ ?

**Votre réponse :**



## 1.4 Trois cibles, une seule inversion

Rien n'oblige la cible à être un seul nombre. Pour prédire $K$ valeurs à la fois, chaque sortie
reçoit sa colonne de paramètres, $W \in \mathbb{R}^{(n+1) \times K}$, et les cibles s'empilent en
lignes de $Y \in \mathbb{R}^{m \times K}$ :

$$\boxed{\ h_W(x) = W^T x \in \mathbb{R}^K, \qquad W = (X^T X)^{-1} X^T Y\ }$$

`load_linnerud` donne 20 sportifs, trois exercices en entrée (tractions, abdominaux, sauts) et
trois mesures physiologiques en sortie (poids, tour de taille, pouls).

**Q6.** Dans $W = (X^T X)^{-1} X^T Y$, le facteur $(X^T X)^{-1}$ ne dépend pas de $Y$. Que
coûtent alors les trois régressions, comparées à une seule ?

**Votre réponse :**



In [ ]:
linnerud = load_linnerud(as_frame=True)
X_sport, Y_sport = linnerud.data.values, linnerud.target.values
print(f"{X_sport.shape[0]} sportifs, entrées {list(linnerud.data.columns)}, "
      f"sorties {list(linnerud.target.columns)}")

# TODO: solve the K regressions at once, W of shape (n + 1, K)
W = ...

# the same K regressions, fitted one target at a time
W_separate = np.column_stack([normal_equation(X_sport, Y_sport[:, k])
                              for k in range(Y_sport.shape[1])])
assert W.shape == (4, 3)
assert np.allclose(W, W_separate)
assert np.allclose(W[0], LinearRegression().fit(X_sport, Y_sport).intercept_)

print()
print(pd.DataFrame(W.round(3), index=["biais"] + list(linnerud.data.columns),
                   columns=list(linnerud.target.columns)).to_string())

**Q7.** Le tableau donne les coefficients des trois exercices pour `Weight`, ajustés sur vingt
sportifs. Ces coefficients sont ceux du maximum de vraisemblance : que faut-il en craindre, avec
vingt lignes seulement ?

**Votre réponse :**

